# Titanic Survival Classification

В этом проекте рассматривается задача классификации на датасете Titanic.  
Цель: по признакам пассажира предсказать, выжил он или нет.

## Что делается в ноутбуке
- загрузка и первичный обзор данных;
- обработка пропусков;
- выделение числовых и категориальных признаков;
- preprocessing через `ColumnTransformer`;
- обучение дерева решений в `Pipeline`;
- подбор параметров и оценка качества модели.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.metrics import classification_report

%matplotlib inline

df = pd.read_csv(
    "https://github.com/Ultraluxe25/Karpov-Stepik-Introduction-to-DS-and-ML/raw/main/csv/titanic.csv"
)

df.head()

## 1. Первичный обзор данных
Сначала посмотрим на структуру таблицы, признаки и пропуски.

In [ ]:
df.shape

In [ ]:
df.info()

In [ ]:
df.isnull().sum()

## 2. Подготовка признаков
Для модели используем несколько базовых признаков:
- `Pclass`
- `Sex`
- `Embarked`
- `Parch`
- `Age`
- `SibSp`
- `Fare`

In [ ]:
features = ["Pclass", "Sex", "Embarked", "Parch", "Age", "SibSp", "Fare"]
target = "Survived"

X = df[features].copy()
y = df[target].copy()

X.head()

## 3. Разделение на train/test

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

X_train.shape, X_test.shape

## 4. Preprocessing
Категориальные признаки кодируются через `OneHotEncoder`,  
числовые — через заполнение пропусков и масштабирование.

In [ ]:
categorical_features = ["Pclass", "Sex", "Embarked"]
numeric_features = ["Parch", "Age", "SibSp", "Fare"]

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

preprocessor = ColumnTransformer(
    transformers=[
        ("cat", categorical_transformer, categorical_features),
        ("num", numeric_transformer, numeric_features)
    ]
)

preprocessor

## 5. Модель
В качестве baseline используется дерево решений внутри `Pipeline`.

In [ ]:
pipeline_dt = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", DecisionTreeClassifier(random_state=42))
])

param_grid = {
    "model__max_depth": [2, 3, 4, 5, 6, 8, 10],
    "model__min_samples_split": [2, 5, 10, 20],
    "model__min_samples_leaf": [1, 2, 5, 10]
}

grid_search = GridSearchCV(
    estimator=pipeline_dt,
    param_grid=param_grid,
    cv=5,
    scoring="f1",
    n_jobs=-1
)

grid_search.fit(X_train, y_train)

grid_search.best_params_

In [ ]:
best_model = grid_search.best_estimator_

y_pred = best_model.predict(X_test)

print(classification_report(y_test, y_pred, zero_division=0))

## 6. Визуализация дерева
Для наглядности отдельно построим дерево из лучшей модели.

In [ ]:
plt.figure(figsize=(20, 10))
plot_tree(
    best_model.named_steps["model"],
    filled=True,
    rounded=True,
    fontsize=8
)
plt.show()

## Выводы

1. Для задачи классификации Titanic достаточно небольшого набора базовых признаков, чтобы получить рабочую baseline-модель.
2. `Pipeline` и `ColumnTransformer` позволяют удобно объединить preprocessing и обучение модели в единый воспроизводимый процесс.
3. Дерево решений даёт интерпретируемую модель, на которой можно анализировать логику разбиений.
4. Подбор гиперпараметров через `GridSearchCV` помогает улучшить качество по сравнению с моделью с параметрами по умолчанию.
5. Проект можно развить дальше за счёт сравнения с логистической регрессией, Random Forest или градиентным бустингом.